In [8]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("/workspaces/SebastianBot")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
from sebastian.clients.deepseek.credentials import DeepSeekApiKey

c = DeepSeekApiKey(api_key="YOUR_DEEPSEEK_API_KEY")

In [10]:
from cloud.helper.secrets import get_secret, SecretKeys


credentials = get_secret(SecretKeys.DeepSeekCredentials)

In [11]:
from pydantic import BaseModel, Field


class Joke(BaseModel):
    genre: str = Field(description="The genre of the joke, e.g., 'programming', 'knock-knock', etc.")
    joke: str = Field(description="The content of the joke.")


In [12]:
import requests

url = "https://api.deepseek.com/chat/completions"

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {credentials.api_key}",
}

payload = {
    "model": "deepseek-v4-flash",
    "messages": [
        {"role": "system", "content": f"Return only valid JSON with the schema {Joke.model_json_schema()}"},
        {"role": "user", "content": "Tell me a joke about programmers. Set genre to programming."},
    ],
    "thinking": {"type": "enabled"},
    "reasoning_effort": "high",
    "response_format": {"type": "json_object"},
    "stream": False,
}

response = requests.post(url, headers=headers, json=payload, timeout=60)
print(response.status_code)
print(response.json())

200
{'id': '42fd83a3-a32e-442a-a191-249844d7370a', 'object': 'chat.completion', 'created': 1786824238, 'model': 'deepseek-v4-flash', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '{"genre":"programming","joke":"Why do programmers prefer dark mode? Because light attracts bugs."}', 'reasoning_content': 'We need to output JSON with genre "programming" and a joke. The joke can be a classic programmer joke. Let\'s craft one. Ensure JSON is valid. Only JSON, no other text. Use double quotes. The joke content can include characters that need escaping if any, but likely fine. Let\'s do: Why do programmers prefer dark mode? Because light attracts bugs. That\'s a common one. Or another. Let\'s use that. Then construct JSON.\n\n{\n  "genre": "programming",\n  "joke": "Why do programmers prefer dark mode? Because light attracts bugs."\n}\n\nMake sure no trailing commas.'}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 219, 'completion_tokens': 152, 

In [13]:
response_data = response.json()
response_text = response_data["choices"][0]["message"]["content"]
joke = Joke.model_validate_json(response_text)
joke

Joke(genre='programming', joke='Why do programmers prefer dark mode? Because light attracts bugs.')

In [17]:
from sebastian.clients.deepseek.client import DeepSeekClient


client = DeepSeekClient(credentials)

In [18]:
client.get_response(prompt="Tell me a joke about programmers. Set genre to programming.", response_schema=Joke)

Joke(genre='programming', joke='Why do programmers prefer dark mode? Because light attracts bugs.')